# Data Visualization — Quiz (Lectures 1–7)

**Time:** 90 minutes
**Tools:** Plotly only (`plotly.express` first, then `update_traces` / `update_layout` / `update_xaxes` / `update_yaxes`)

**Your task:** answer each analytical question with **one polished, explanatory visualization**. For every question **you decide which chart type fits best** — and justify the choice in one or two sentences.

**Every figure must have:**
1. An **insight title** that states the finding, not just the variables
2. `template='plotly_white'` and a decluttered layout (no unnecessary gridlines, legends or axis titles)
3. A **CVD-safe palette** — no red/green; grey for context + one accent colour where it helps
4. Colour that **encodes meaning**, not decoration
5. Direct labels or annotations where they help the reader

**Chart types covered in the course** — choose from these:
line chart · sorted bar chart · stacked / 100% stacked bar chart · scatter plot · bubble chart · histogram · box plot · violin plot · choropleth map · treemap

> 💡 Each question has one chart type that fits it best, and the questions together cover most of the list above. If two of your answers use the same chart type, check the question again.


**Submit:** this notebook pushed to your personal repo as `quiz/dataviz_quiz.ipynb`

## The dataset — World Population 1970–2022

**Source:** [World Population Dataset](https://www.kaggle.com/datasets/iamsouravbanerjee/world-population-dataset) on Kaggle, compiled from Worldometer (UN World Population Prospects estimates). **234 rows × 16 columns — one row per country or territory, no missing values.**

| Column | Type | Description |
|---|---|---|
| `Country`, `Capital` | categorical | Country / territory and its capital |
| `Continent` | categorical | Africa, Asia, Europe, North America, Oceania, South America |
| `CCA3` | spatial | ISO-3 country code — use with `locationmode='ISO-3'` |
| `1970 Population` … `2022 Population` | temporal | Population in 1970, 1980, 1990, 2000, 2010, 2015, 2020, 2022 |
| `Area` | numerical | Land area in km² |
| `Density` | numerical | People per km² (2022) |
| `World share (%)` | numerical | Share of world population in 2022 |
| `Rank` | numerical | Rank by 2022 population |

**The next setup cell also gives you `df_long` — the same data in **tidy (long) format** with one row per country per year. Use `df` for one-year questions and `df_long` for questions about change over time**

In [6]:
import pandas as pd
import plotly.express as px
import numpy as np


df = pd.read_csv('/content/sample_data/world_population.csv')

# Shorter column names
df = df.rename(columns={'Country/Territory': 'Country', 'Area (km²)': 'Area',
                        'Density (per km²)': 'Density', 'World Population Percentage': 'World share (%)'})

# Tidy (long) version: one row per country per year
year_cols = [c for c in df.columns if c.endswith('Population')]
df_long = df.melt(id_vars=['CCA3', 'Country', 'Continent'], value_vars=year_cols,
                  var_name='Year', value_name='Population')

df_long['Year'] = df_long['Year'].str[:4].astype(int)     # '2022 Population' → 2022

print(df.shape, df_long.shape)


(234, 16) (1872, 5)


---
## Q1 — Which continent is growing fastest?

How has the population of each continent **grown since 1970, relative to its 1970 size** — and which continent is growing fastest?


**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*YOUR ANSWER HERE*  I chose a line chart because it is effective for comparing population trends over time. I normalized each continent's population to a base value of 100 in 1970 so that relative growth can be compared fairly. Different CVD-safe colors distinguish the continents.

In [7]:

# Q1: Population growth by continent

# Calculate total population for each continent and year
continent_pop = (
    df_long.groupby(['Continent', 'Year'])['Population']
    .sum()
    .reset_index()
)

# Sort years and calculate relative growth
continent_pop = continent_pop.sort_values('Year')

continent_pop['Growth Index'] = (
    continent_pop['Population'] /
    continent_pop.groupby('Continent')['Population'].transform('first')
) * 100

# Create line chart
fig = px.line(
    continent_pop,
    x='Year',
    y='Growth Index',
    color='Continent',
    markers=True,
    title='Africa has grown fastest since 1970',
    color_discrete_sequence=[
        '#0072B2', '#E69F00', '#CC79A7',
        '#56B4E9', '#666666', '#9467BD'
    ]
)

# Improve chart appearance
fig.update_layout(
    template='plotly_white',
    xaxis_title='Year',
    yaxis_title='Population index (1970 = 100)',
    legend_title=None
)

fig.update_xaxes(showgrid=False)
fig.update_yaxes(showgrid=False)

fig.show()


---
## Q2 — Which countries added the most people?

Which **15 countries added the most people** between 1970 and 2022 — and how many of them are in **Africa**?

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*YOUR ANSWER HERE* I chose a horizontal bar chart to compare the 15 countries with the largest absolute population increases between 1970 and 2022. Horizontal bars make country names easier to read. The bars are sorted by

In [8]:

# Q2: Top 15 countries by population increase

# Calculate population increase from 1970 to 2022
df['Population Increase'] = (
    df['2022 Population'] - df['1970 Population']
)

# Select top 15 countries
top15 = df.nlargest(15, 'Population Increase').copy()

# Highlight African countries
top15['Region'] = top15['Continent'].apply(
    lambda x: 'Africa' if x == 'Africa' else 'Other'
)

# Count African countries
africa_count = (top15['Continent'] == 'Africa').sum()

print("Number of African countries in top 15:", africa_count)

# Create horizontal bar chart
fig = px.bar(
    top15.sort_values('Population Increase'),
    x='Population Increase',
    y='Country',
    color='Region',
    orientation='h',
    title='Top 15 countries by population increase (1970–2022)',
    color_discrete_map={
        'Africa': '#E69F00',
        'Other': '#0072B2'
    }
)

fig.update_layout(
    template='plotly_white',
    xaxis_title='Population increase (people)',
    yaxis_title=None,
    legend_title=None
)

fig.show()


Number of African countries in top 15: 4


In [10]:
fig.update_yaxes(categoryorder='total ascending')

**Conclusion:** India experienced the largest population increase between 1970 and 2022, followed by China. Among the 15 countries with the largest increases, four are African: Nigeria, Ethiopia, DR Congo, and Egypt. This shows that significant absolute population growth occurred across both Asia and Africa.

---
## Q3 — Do bigger countries have more people?

Is there a relationship between a country's **land area** and its **2022 population** — does it differ by **continent**, and which countries clearly break the pattern?

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*YOUR ANSWER HERE* I chose a scatter plot to explore the relationship between land area and population in 2022. Each point represents a country, and different colors distinguish continents. Logarithmic scales make it easier to compare countries with very different sizes and populations.

The chart helps identify whether larger countries tend to have larger populations. It also highlights outliers, such as countries with large land areas but relatively small populations, or small land areas with very high populations.

In [9]:

# Q3: Relationship between land area and population

# Remove rows with zero or missing values
df_scatter = df[
    (df['Area'] > 0) &
    (df['2022 Population'] > 0)
].copy()

# Create scatter plot
fig = px.scatter(
    df_scatter,
    x='Area',
    y='2022 Population',
    color='Continent',
    hover_name='Country',
    log_x=True,
    log_y=True,
    title='Country land area vs population (2022)',
    labels={
        'Area': 'Land area (km²)',
        '2022 Population': 'Population in 2022'
    },
    color_discrete_sequence=[
        '#0072B2', '#E69F00', '#009E73',
        '#CC79A7', '#56B4E9', '#666666'
    ]
)

fig.update_layout(
    template='plotly_white',
    legend_title='Continent'
)

fig.show()


**Conclusion:** There is a general positive relationship between land area and population in 2022, but the relationship varies considerably across countries. India and China have exceptionally large populations, while Australia and Canada have large land areas but relatively small populations. Singapore is another notable outlier because it has a small land area but a relatively large population. Therefore, land area alone does not determine population size.

---
## Q4 — How crowded are the continents?

How does **population density vary within and across continents**? Show the spread, not just the average.

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*YOUR ANSWER HERE* I chose a box plot with individual data points to compare population density across continents. A box plot shows the median, interquartile range, and spread of population density within each continent, while the individual points reveal country-level differences and outliers.

I used a logarithmic scale because population densities vary greatly between countries. Different colors distinguish continents, making comparisons easier. This visualization shows both typical density levels and the variation within each continent rather than focusing only on averages.

In [11]:

# Q4: Population density across continents

# Filter countries with valid population density
df_density = df[
    (df['Density'] > 0)
].copy()

# Create box plot with individual country points
fig = px.box(
    df_density,
    x='Continent',
    y='Density',
    color='Continent',
    points='all',
    hover_name='Country',
    log_y=True,
    title='Population density varies widely within continents',
    labels={
        'Continent': 'Continent',
        'Density': 'Population density (people/km²)'
    },
    color_discrete_sequence=[
        '#0072B2', '#E69F00', '#009E73',
        '#CC79A7', '#56B4E9', '#666666'
    ]
)

# Improve appearance
fig.update_layout(
    template='plotly_white',
    xaxis_title='Continent',
    yaxis_title='Population density (people/km², log scale)',
    showlegend=False
)

fig.update_xaxes(showgrid=False)

fig.show()


In [12]:
fig.update_yaxes(
    tickvals=[0.1, 1, 10, 100, 1000, 10000],
    ticktext=['0.1', '1', '10', '100', '1k', '10k']
)

**Conclusion:** Population density differs substantially both within and across continents. North America shows a high median density and wide variation in this dataset, while South America has a relatively low median. Several continents contain countries with extremely high or low densities. The box plots demonstrate why comparing distributions provides more information than comparing averages alone.

---
## Q5 — Where did populations grow — and where did they shrink?

Which countries' populations **grew and which shrank between 2000 and 2022** (percentage change), and where in the world are they?

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*YOUR ANSWER HERE* **Chart choice & justification:**

I chose a choropleth world map to visualize the percentage change in population between 2000 and 2022. This chart allows us to identify geographical patterns in population growth and decline across countries.

I used a diverging color scale, with blue representing population growth and red representing population decline. A neutral color represents little or no change. The map makes it easy to compare countries geographically and identify regions experiencing different population trends.

Percentage change was used instead of absolute population change to allow fair comparisons between countries with different population sizes.

In [13]:

# Q5: Population growth and decline (2000–2022)

# Create a copy of the dataset
df_map = df.copy()

# Calculate percentage population change
df_map['Population Change (%)'] = (
    (df_map['2022 Population'] -
     df_map['2000 Population']) /
    df_map['2000 Population']
) * 100

# Remove invalid values
df_map = df_map[
    (df_map['2000 Population'] > 0) &
    np.isfinite(df_map['Population Change (%)'])
].copy()

# Create choropleth world map
fig = px.choropleth(
    df_map,
    locations='CCA3',
    color='Population Change (%)',
    hover_name='Country',
    hover_data={
        'Population Change (%)': ':.1f',
        'Continent': True,
        'CCA3': False
    },
    color_continuous_scale='RdBu',
    color_continuous_midpoint=0,
    title='Population change by country (2000–2022)',
    labels={
        'Population Change (%)': 'Population change (%)'
    }
)

# Improve map appearance
fig.update_geos(
    showcoastlines=True,
    coastlinecolor='gray',
    showframe=False,
    projection_type='natural earth'
)

fig.update_layout(
    template='plotly_white',
    coloraxis_colorbar_title='Change (%)',
    margin=dict(l=10, r=10, t=60, b=10)
)

fig.show()


In [14]:

# Show countries where population declined
declining = df_map[
    df_map['Population Change (%)'] < 0
]

print("Countries with declining populations:")
print(
    declining[
        ['Country', 'Population Change (%)']
    ].sort_values('Population Change (%)').to_string(index=False)
)


Countries with declining populations:
                         Country  Population Change (%)
        Northern Mariana Islands             -38.321840
                  American Samoa             -23.968745
                       Lithuania             -23.601880
                Marshall Islands             -23.338374
                         Moldova             -23.016822
                          Latvia             -22.648786
          Bosnia and Herzegovina             -22.630888
                    Vatican City             -21.658986
               Wallis and Futuna             -21.401888
                         Ukraine             -18.776723
                        Bulgaria             -16.248311
                      Martinique             -15.035731
                     Puerto Rico             -15.016587
                      Montserrat             -14.558194
                         Armenia             -12.247157
                         Georgia             -12.210223
          

**Conclusion:** Population change between 2000 and 2022 varied significantly across the world. Many countries experienced population growth, particularly in Africa, while 34 countries and territories in the dataset experienced population decline. The Northern Mariana Islands recorded the largest percentage decline at approximately 38.32%, followed by American Samoa and Lithuania. Several Eastern European countries also experienced population declines. The choropleth map effectively highlights these geographical differences in population trends.

---
## Q6 — Where do the world's 8 billion people live?

How is the **2022 world population** distributed across **continents and the countries within them**? Show both levels in one figure.

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*YOUR ANSWER HERE* **Chart choice & justification:**

I chose a sunburst chart because it effectively displays hierarchical data at two levels: continents and countries. The inner ring represents continents, while the outer ring represents individual countries within each continent.

The size of each segment is proportional to its 2022 population, making it easy to compare the distribution of the world's population. Different colors distinguish continents, and the interactive chart allows users to explore country-level population values.

This visualization shows both the overall distribution across continents and the contributions of individual countries in a single figure.

In [16]:

# Q6: World population distribution in 2022

# Prepare the data
df_sunburst = df[
    (df['2022 Population'] > 0)
].copy()

# Create sunburst chart
fig = px.sunburst(
    df_sunburst,
    path=['Continent', 'Country'],
    values='2022 Population',
    color='Continent',
    title='Where do the world’s 8 billion people live? (2022)',
    color_discrete_map={
        'Asia': '#0072B2',
        'Africa': '#009E73',
        'Europe': '#E69F00',
        'North America': '#CC79A7',
        'South America': '#56B4E9',
        'Oceania': '#666666'
    }
)

# Improve chart appearance
fig.update_layout(
    template='plotly_white',
    margin=dict(l=10, r=10, t=60, b=10),
    height=700
)

fig.update_traces(
    textinfo='label+percent parent',
    hovertemplate=(
        '<b>%{label}</b><br>'
        'Population: %{value:,.0f}<br>'
        'Share of parent: %{percentParent:.1%}'
        '<extra></extra>'
    )
)

fig.show()


**Conclusion:** The sunburst chart shows that Asia contains the largest share of the world's population in 2022, accounting for approximately 59%, followed by Africa with 18%. India and China are the largest contributors to Asia's population, each representing approximately 30% of the continent's total. Oceania has the smallest share of the global population. The chart effectively displays both continental and country-level population distributions in one visualization.

---
## Q7 — Whose share of the world is rising?

How have the shares of world population held by **Africa, Asia and Europe** changed from 1970 to 2022? Group the other three continents as `'Rest of the world'`.

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*YOUR ANSWER HERE* **Chart choice & justification:**

I chose a 100% stacked area chart to show how the shares of the world population held by Africa, Asia, Europe, and the Rest of the world changed between 1970 and 2022.

This visualization is appropriate because it shows changes in the relative contribution of each group over time. Each year's total equals 100%, making it easy to compare population shares rather than absolute population sizes.

I used distinct, color-vision-friendly colors to distinguish the four groups. The chart clearly highlights which continents gained or lost a share of the global population.

In [17]:

# Q7: Changes in world population shares (1970–2022)

# Create a copy of the long-format dataset
df_q7 = df_long.copy()

# Group the other three continents together
df_q7['Group'] = df_q7['Continent'].apply(
    lambda x: x if x in ['Africa', 'Asia', 'Europe']
    else 'Rest of the world'
)

# Calculate population for each group and year
group_pop = (
    df_q7.groupby(['Year', 'Group'])['Population']
    .sum()
    .reset_index()
)

# Calculate total world population for each year
group_pop['World Population'] = (
    group_pop.groupby('Year')['Population']
    .transform('sum')
)

# Calculate population percentage share
group_pop['Share (%)'] = (
    group_pop['Population'] /
    group_pop['World Population']
) * 100

# Create stacked area chart
fig = px.area(
    group_pop,
    x='Year',
    y='Share (%)',
    color='Group',
    category_orders={
        'Group': [
            'Africa',
            'Asia',
            'Europe',
            'Rest of the world'
        ]
    },
    title='Changing shares of world population (1970–2022)',
    color_discrete_map={
        'Africa': '#009E73',
        'Asia': '#0072B2',
        'Europe': '#E69F00',
        'Rest of the world': '#CC79A7'
    }
)

# Improve chart appearance
fig.update_layout(
    template='plotly_white',
    xaxis_title='Year',
    yaxis_title='Share of world population (%)',
    legend_title='Region',
    yaxis=dict(range=[0, 100]),
    hovermode='x unified'
)

fig.update_xaxes(showgrid=False)
fig.update_yaxes(ticksuffix='%')

fig.show()


**Conclusion:** The 100% stacked area chart shows how the distribution of the world's population changed between 1970 and 2022. Africa's share increased from approximately 10% to 18%, while Europe's share declined from approximately 18% to 9%. Asia remained the largest population group, accounting for around 59% of the global population in 2022. The Rest of the world's combined share remained relatively stable. This demonstrates a significant shift in the global population distribution over time.